# Brain Tumor MRI — EfficientNetB0 + Grad-CAM
**Dataset:** Nickparvar | 4 clases: glioma, meningioma, notumor, pituitary  
**Arquitectura:** EfficientNetB0 con fine-tuning en 2 fases

**Fixes aplicados:**
- Sin doble rescale: generadores en 0-255, EfficientNet escala internamente
- val_gen con generador separado sin augmentation
- BatchNormalization de EfficientNet congelada en fase 2
- Bug de crop corregido en preprocess_mri
- Warmup de 2 epocas antes del fine-tuning

In [ ]:
# ================================================================
# CELDA 1 — Subir dataset desde PC y extraer
# ================================================================
from google.colab import files
import zipfile, os

print('Sube el archivo archive.zip con el dataset...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/dataset')
print('Dataset extraido en /content/dataset')

for split in ['Training', 'Testing']:
    for cls in ['glioma', 'meningioma', 'notumor', 'pituitary']:
        p = f'/content/dataset/{split}/{cls}'
        n = len(os.listdir(p)) if os.path.exists(p) else 0
        print(f'  {split}/{cls}: {n} imagenes')

In [ ]:
# ================================================================
# CELDA 2 — Instalar dependencias y verificar GPU
# ================================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'opencv-python-headless'], check=True)

import os, cv2, numpy as np, tensorflow as tf
import matplotlib.pyplot as plt, seaborn as sns
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import (EarlyStopping,
                                         ReduceLROnPlateau,
                                         ModelCheckpoint)
from sklearn.metrics import classification_report, confusion_matrix
from scipy.ndimage import gaussian_filter

print(f'TF {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ================================================================
# CELDA 3 — Configuracion global
# ================================================================
IMG_SIZE    = 224
BATCH       = 32
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
DATA_DIR    = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset'
CLEAN_DIR   = '/tmp/brain_clean'
CKPT_PATH   = 'best_brain_model.keras'
tf.random.set_seed(42)
np.random.seed(42)
print('Configuracion lista')

In [ ]:
# ================================================================
# CELDA 4 — Preprocesamiento: crop automatico de margenes negros
# FIX: x2 = min(W, x + w + pad)  —  antes era shape[1] + w (bug)
# ================================================================
def preprocess_mri(img_bgr, size=IMG_SIZE):
    if img_bgr is None:
        return None
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        margin_ratio = 1 - (w * h) / (img_bgr.shape[1] * img_bgr.shape[0])
        if margin_ratio > 0.05:
            pad = 8
            x1 = max(0, x - pad)
            y1 = max(0, y - pad)
            x2 = min(img_bgr.shape[1], x + w + pad)  # FIX: x + w, no shape[1] + w
            y2 = min(img_bgr.shape[0], y + h + pad)
            img_bgr = img_bgr[y1:y2, x1:x2]
    return cv2.resize(img_bgr, (size, size), interpolation=cv2.INTER_LANCZOS4)


def build_clean_dataset(data_dir, class_names, clean_dir):
    if os.path.exists(clean_dir):
        import shutil
        shutil.rmtree(clean_dir)
    for split in ['Training', 'Testing']:
        for cls in class_names:
            src = os.path.join(data_dir, split, cls)
            dst = os.path.join(clean_dir, split, cls)
            os.makedirs(dst, exist_ok=True)
            if not os.path.exists(src):
                print(f'No existe: {src}'); continue
            for fname in os.listdir(src):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img = cv2.imread(os.path.join(src, fname))
                    processed = preprocess_mri(img)
                    if processed is not None:
                        cv2.imwrite(os.path.join(dst, fname), processed)
    print('Preprocesamiento completado')


build_clean_dataset(DATA_DIR, CLASS_NAMES, CLEAN_DIR)

In [ ]:
# ================================================================
# CELDA 5 — Generadores de datos
# FIX 1: sin rescale=1./255 — EfficientNet escala internamente (0-255)
# FIX 2: val_gen usa generador separado sin augmentation
# ================================================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator

kw = dict(
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical',
    classes=CLASS_NAMES
)

# Augmentation suave para MRI — sin rescale
train_datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.10,
    width_shift_range=0.07,
    height_shift_range=0.07,
    brightness_range=[0.85, 1.15],
    fill_mode='constant',
    cval=0,
    validation_split=0.15
)

# FIX: val con generador limpio, sin augmentation
val_datagen = ImageDataGenerator(validation_split=0.15)

train_gen = train_datagen.flow_from_directory(
    os.path.join(CLEAN_DIR, 'Training'),
    subset='training', shuffle=True, seed=42, **kw)

val_gen = val_datagen.flow_from_directory(
    os.path.join(CLEAN_DIR, 'Training'),
    subset='validation', shuffle=False, seed=42, **kw)

test_gen = ImageDataGenerator().flow_from_directory(
    os.path.join(CLEAN_DIR, 'Testing'),
    shuffle=False, **kw)

print(f'Clases : {train_gen.class_indices}')
print(f'Train  : {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}')

# Verificacion: rango debe ser 0-255
sx, _ = next(train_gen)
print(f'Rango batch: [{sx.min():.0f}, {sx.max():.0f}]  <- debe ser 0-255')

In [ ]:
# ================================================================
# CELDA 6 — Modelo EfficientNetB0
# FIX: sin capa Rescaling adicional
# EfficientNetB0 incluye preproceso interno que espera 0-255
# ================================================================
def build_model(num_classes=4):
    base = EfficientNetB0(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_img')
    x = base(inputs, training=False)  # recibe 0-255 directamente
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='bn_head')(x)
    x = layers.Dense(256, activation='relu', name='dense_256',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4, name='drop_1')(x)
    x = layers.Dense(128, activation='relu', name='dense_128',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.2, name='drop_2')(x)
    outputs = layers.Dense(num_classes, activation='softmax',
                           name='predictions')(x)
    return Model(inputs, outputs, name='brain_classifier'), base


model, base = build_model()
model.summary(line_length=90)

# Verificar forward pass con imagen 0-255
test_input = np.ones((1, 224, 224, 3), dtype=np.float32) * 128.0
out = model(test_input, training=False)
print(f'Forward pass OK — suma probs: {out.numpy().sum():.4f}  (debe ser ~1.0)')

In [ ]:
# ================================================================
# CELDA 7 — Entrenamiento Fase 1 (head, base congelada)
# ================================================================
metrics = [
    'accuracy',
    tf.keras.metrics.AUC(name='auc'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=metrics
)

print('\n' + '='*50)
print('FASE 1: Head (base congelada)')
print('='*50)

h1 = model.fit(
    train_gen, epochs=20, validation_data=val_gen,
    callbacks=[
        EarlyStopping('val_accuracy', patience=6,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau('val_loss', factor=0.5, patience=3,
                          min_lr=1e-7, verbose=1)
    ], verbose=1
)

best_v1 = max(h1.history['val_accuracy'])
print(f'\nMejor val_accuracy fase 1: {best_v1:.4f}')
if   best_v1 < 0.70: print('< 70% -- revisar preprocesado')
elif best_v1 < 0.85: print(f'{best_v1:.2%} -- aceptable, fine-tuning mejorara')
else:                 print(f'{best_v1:.2%} -- excelente')

In [ ]:
# ================================================================
# CELDA 8 — Entrenamiento Fase 2 (fine-tuning ultimas 40 capas)
# FIX: BatchNormalization de EfficientNet permanece congelada
# FIX: warmup con doble compile (compatible con Keras 3)
# ================================================================
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

# CRITICO: mantener BatchNorm en modo inferencia
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

frozen    = sum(1 for l in base.layers if not l.trainable)
trainable = sum(1 for l in base.layers if l.trainable)
print(f'Capas base — congeladas: {frozen} | entrenables: {trainable}')

# Warmup: doble compile para evitar set_value (incompatible con Keras 3)
print('\nWarmup (2 epocas a LR=1e-6)...')
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-6),
    loss='categorical_crossentropy',
    metrics=metrics
)
model.fit(train_gen, epochs=2, validation_data=val_gen, verbose=1)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=metrics
)
print('LR restaurado a 1e-5 via recompile')

print('\n' + '='*50)
print('FASE 2: Fine-tuning (ultimas 40 capas)')
print('='*50)

h2 = model.fit(
    train_gen, epochs=50, validation_data=val_gen,
    callbacks=[
        EarlyStopping('val_accuracy', patience=8,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau('val_loss', factor=0.5, patience=4,
                          min_lr=1e-8, verbose=1),
        ModelCheckpoint(CKPT_PATH, monitor='val_accuracy',
                        save_best_only=True, verbose=1)
    ], verbose=1
)

best_v2 = max(h2.history['val_accuracy'])
print(f'\nMejor val_accuracy fase 2: {best_v2:.4f}')
if   best_v2 < 0.80: print('< 80% -- revisar bugs de preprocesado')
elif best_v2 < 0.85: print(f'{best_v2:.2%} -- aceptable')
else:                 print(f'{best_v2:.2%} -- excelente para produccion')

In [ ]:
# ================================================================
# CELDA 9 — Curvas de entrenamiento
# ================================================================
acc   = h1.history['accuracy']      + h2.history['accuracy']
vacc  = h1.history['val_accuracy']  + h2.history['val_accuracy']
loss  = h1.history['loss']          + h2.history['loss']
vloss = h1.history['val_loss']      + h2.history['val_loss']
ft    = len(h1.history['accuracy'])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(acc,  label='Train')
ax[0].plot(vacc, label='Validation')
ax[0].axvline(ft, color='gray', ls='--', label='Fine-tuning')
ax[0].set_title('Accuracy'); ax[0].set_ylim(0, 1)
ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(loss,  label='Train')
ax[1].plot(vloss, label='Validation')
ax[1].axvline(ft, color='gray', ls='--')
ax[1].set_title('Loss'); ax[1].legend(); ax[1].grid(alpha=.3)

plt.suptitle('Curvas de entrenamiento — EfficientNetB0')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# ================================================================
# CELDA 10 — Evaluacion en test set
# ================================================================
test_gen.reset()
res = model.evaluate(test_gen, verbose=1)
print(f'\nTest Accuracy : {res[1]:.4f}')
print(f'Test AUC      : {res[2]:.4f}')
print(f'Test Precision: {res[3]:.4f}')
print(f'Test Recall   : {res[4]:.4f}')

test_gen.reset()
y_pred = np.argmax(model.predict(test_gen, verbose=0), axis=1)
y_true = test_gen.classes
print('\n' + classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Confusion Matrix — Accuracy: {res[1]:.2%}')
plt.ylabel('Real'); plt.xlabel('Predicho')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ================================================================
# CELDA 11 — Grad-CAM compatible con Keras 3 + TF 2.19
#
# Problema en Keras 3: acceder a layer.output de capas internas
# de un modelo anidado (EfficientNetB0) desde el grafo del modelo
# padre lanza "Graph disconnected" porque son grafos distintos.
#
# Solucion en 2 pasos:
#   1. eff_multi: modelo multi-salida construido DENTRO del grafo
#      propio de EfficientNetB0 (conv_out + eff.output).
#   2. grad_model: nuevo modelo funcional que encadena eff_multi
#      con las capas head, formando UN UNICO grafo.
# Con ambos tensores en el mismo grafo, tape.gradient(loss, conv_out)
# funciona sin tape.watch() explicito (patron oficial Keras Grad-CAM).
# ================================================================

def _find_conv_layer(eff):
    """Devuelve el nombre de la ultima Conv2D de EfficientNetB0."""
    for name in ['top_conv', 'block7a_project_conv']:
        try:
            eff.get_layer(name)
            return name
        except ValueError:
            continue
    for layer in reversed(eff.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise RuntimeError('No Conv2D encontrada en EfficientNetB0')


def build_gradcam_model(full_model):
    """
    Construye el modelo de Grad-CAM compatible con Keras 3.

    Paso 1 — eff_multi: modelo multi-salida dentro del grafo propio
    de EfficientNetB0. Produce [conv_out, eff_features] desde la
    entrada de EfficientNet. Esto no cruza grafos y no genera
    el error "Graph disconnected" de Keras 3.

    Paso 2 — grad_model: nuevo grafo funcional que llama a eff_multi
    y luego aplica las capas head del modelo original.
    conv_out y predictions quedan en el mismo grafo, lo que permite
    que tape.gradient(loss, conv_out) funcione correctamente.
    """
    eff = full_model.get_layer('efficientnetb0')
    conv_name = _find_conv_layer(eff)
    print(f"Capa Grad-CAM: '{conv_name}' "
          f"shape: {eff.get_layer(conv_name).output_shape}")

    # Paso 1: multi-salida dentro del grafo de EfficientNet
    eff_multi = tf.keras.Model(
        inputs=eff.inputs,
        outputs=[eff.get_layer(conv_name).output, eff.output],
        name='eff_multi'
    )

    # Paso 2: encadenar eff_multi + head en un unico grafo funcional
    img_in = tf.keras.Input(
        shape=(IMG_SIZE, IMG_SIZE, 3), name='gradcam_input')
    conv_out, eff_feat = eff_multi(img_in, training=False)
    x = eff_feat
    for lname in ['gap', 'bn_head', 'dense_256', 'drop_1',
                  'dense_128', 'drop_2', 'predictions']:
        x = full_model.get_layer(lname)(x)

    return tf.keras.Model(
        inputs=img_in,
        outputs=[conv_out, x],
        name='gradcam_model'
    )


def gradcam_heatmap(grad_model, img_uint8, class_idx=None):
    """
    Calcula el heatmap Grad-CAM para img_uint8 (HWC, 0-255).
    tape.gradient(loss, conv_out) funciona sin tape.watch() porque
    ambos tensores son salidas del mismo grafo funcional y TF registra
    todas las operaciones intermedias dentro del contexto del tape.
    """
    img_t = tf.cast(img_uint8[None], tf.float32)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_t, training=False)
        if class_idx is None:
            class_idx = int(tf.argmax(preds[0]))
        loss = preds[:, class_idx]
    grads = tape.gradient(loss, conv_out)
    if grads is None:
        return np.zeros((IMG_SIZE, IMG_SIZE)), False
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    hm = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
    hm = tf.maximum(hm, 0).numpy()
    if hm.max() < 1e-8:
        return np.zeros((IMG_SIZE, IMG_SIZE)), False
    hm = np.clip(hm / (np.percentile(hm, 99) + 1e-8), 0, 1)
    hm = cv2.resize(hm, (IMG_SIZE, IMG_SIZE),
                    interpolation=cv2.INTER_LANCZOS4)
    return hm, True


def make_overlay(img_uint8, heatmap, alpha=0.5):
    colored = cv2.applyColorMap(
        np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    return cv2.addWeighted(img_uint8, 1 - alpha, colored, alpha, 0)


def show_gradcam_grid(model, grad_model, test_gen,
                      class_names, n_per_class=2):
    """Grid 3 columnas: imagen original | heatmap | overlay."""
    test_gen.reset()
    all_imgs, all_lbls, total = [], [], 0
    for imgs, lbls in test_gen:
        all_imgs.append(imgs); all_lbls.append(lbls)
        total += len(imgs)
        if total >= 400: break
    all_imgs = np.vstack(all_imgs)
    all_true = np.argmax(np.vstack(all_lbls), axis=1)

    n_rows = len(class_names) * n_per_class
    fig, axes = plt.subplots(n_rows, 3, figsize=(12, n_rows * 3.2))

    row = 0
    for cls_i, cls_name in enumerate(class_names):
        idxs = np.where(all_true == cls_i)[0][:30]
        preds_b = model.predict(all_imgs[idxs], verbose=0)
        pred_cls = np.argmax(preds_b, axis=1)
        correct   = idxs[pred_cls == cls_i][:n_per_class]
        incorrect = idxs[pred_cls != cls_i][:n_per_class]
        selected  = list(correct)
        if len(selected) < n_per_class:
            selected += list(incorrect)[:n_per_class - len(selected)]

        for s_i in selected[:n_per_class]:
            img_u8 = all_imgs[s_i].astype(np.uint8)
            pr = model.predict(all_imgs[s_i:s_i+1], verbose=0)[0]
            pc = int(np.argmax(pr)); conf = pr[pc]
            ok    = 'OK' if pc == cls_i else 'X'
            color = 'limegreen' if ok == 'OK' else 'tomato'

            hm, valid = gradcam_heatmap(grad_model, img_u8, pc)
            overlay   = make_overlay(img_u8, hm)
            hm_col    = cv2.applyColorMap(
                np.uint8(255 * hm), cv2.COLORMAP_JET)

            axes[row, 0].imshow(
                cv2.cvtColor(img_u8, cv2.COLOR_BGR2RGB))
            axes[row, 0].set_title(f'Real: {cls_name}', fontsize=9)
            axes[row, 0].axis('off')

            axes[row, 1].imshow(
                cv2.cvtColor(hm_col, cv2.COLOR_BGR2RGB))
            axes[row, 1].set_title(
                'Grad-CAM' if valid else 'Grad-CAM [grad≈0]',
                fontsize=9)
            axes[row, 1].axis('off')

            axes[row, 2].imshow(
                cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
            axes[row, 2].set_title(
                f'{ok} {class_names[pc]} ({conf:.0%})',
                fontsize=9, color=color)
            axes[row, 2].axis('off')
            row += 1

    plt.suptitle('Grad-CAM — EfficientNetB0 (Keras 3)',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('gradcam_final.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Grid guardado: gradcam_final.png')


print('Funciones Grad-CAM (Keras 3) listas')

In [ ]:
# ================================================================
# CELDA 12 — Ejecutar Grad-CAM, guardar modelo y descargar artefactos
# ================================================================
grad_model = build_gradcam_model(model)
print(f'Modelo Grad-CAM construido: {grad_model.name}')

show_gradcam_grid(model, grad_model, test_gen,
                  CLASS_NAMES, n_per_class=2)

model.save('brain_tumor_final.keras')
print('Modelo guardado: brain_tumor_final.keras')

from google.colab import files as colab_files
for f in ['best_brain_model.keras', 'brain_tumor_final.keras',
          'training_curves.png', 'confusion_matrix.png',
          'gradcam_final.png']:
    if os.path.exists(f):
        colab_files.download(f)
        print(f'Descargando {f}')